# Notebook 4: Clustering Benchmarking & Unsupervised Evaluation
**Course:** INT396 - Unsupervised Learning

This notebook conducts the central benchmark experiment:
- Evaluates K-Means, Agglomerative Hierarchical, and DBSCAN
- Benchmarks across Raw features, PCA features, and UMAP features
- Calculates Silhouette, Calinski-Harabasz, Davies-Bouldin, ARI, and NMI

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from src.data_loader import load_dataset
from src.preprocessing import build_preprocessing_pipeline, fit_transform_pipeline
from src.pca import fit_pca, transform_pca
from src.umap_reduction import fit_umap
from src.clustering import run_kmeans, run_agglomerative
from src.evaluation import evaluate_clustering

X, y, meta, bands, wls = load_dataset('potato_dai70_vnir')
pipe = build_preprocessing_pipeline('standard')
X_raw, _, _ = fit_transform_pipeline(pipe, X)

pca, _ = fit_pca(X_raw, n_components=5)
X_pca = transform_pca(pca, X_raw)

_, X_umap = fit_umap(X_raw, n_components=2, n_neighbors=10, min_dist=0.1, random_state=42)

In [ ]:
# Run comparison matrix
spaces = {'Raw Features': X_raw, 'PCA Features': X_pca, 'UMAP Features': X_umap}
algos = {
    'K-Means': lambda feat: run_kmeans(feat, n_clusters=4, random_state=42),
    'Agglomerative': lambda feat: run_agglomerative(feat, n_clusters=4, linkage='ward')
}

records = []
for rep_name, feat in spaces.items():
    for algo_name, func in algos.items():
        _, labels = func(feat)
        res = evaluate_clustering(feat, labels, y_true=y)
        im = res['internal_metrics']
        em = res['external_metrics']
        records.append({
            'Representation': rep_name,
            'Algorithm': algo_name,
            'Silhouette': im['silhouette_score'],
            'Calinski_Harabasz': im['calinski_harabasz_score'],
            'Davies_Bouldin': im['davies_bouldin_score'],
            'ARI': em['adjusted_rand_index'],
            'NMI': em['normalized_mutual_info']
        })

df_bench = pd.DataFrame(records)
print("Benchmark Results:
", df_bench.to_string(index=False))